# Week 3 - Day 3: Agentic RAG

> **Student notebook.** A TODO cell with hints, then a check that prints `PASS` or `NOT YET`.

A model only knows what it learned in training, plus what is in its **context**. It has never seen **our course notes**, so it guesses when you ask about them. **RAG** (retrieval-augmented generation) fixes that: before answering, find the few pieces of your documents that matter, and put **only those** in the context.

```
course notes --> split into chunks --> embed each chunk --> vector store
                                                               |
question --> agent --"search_course_notes"--> the 3 closest chunks --> answer from the chunks
```

**Agentic** RAG means the search is a **tool**: the agent decides **when** to look something up (a greeting needs no search).

Today:
1. **Split** the notes into chunks.
2. **Embed** them with Google's embedding model, and store them in a **vector store**.
3. **The retriever as a tool**: an agent that answers from the notes.
4. **Check the answer** is grounded in the chunks, with an evaluator like Week 2.

The code follows the LangGraph guide **[Agentic RAG](https://docs.langchain.com/oss/python/langgraph/agentic-rag)** and the **[Google embeddings](https://docs.langchain.com/oss/python/integrations/text_embedding/google_generative_ai)** page.

## Setup
Today you need **two** free keys:
- **Groq** (console.groq.com -> API Keys), saved as `GROQ_API_KEY`
- **Google AI Studio** (aistudio.google.com -> Get API key), saved as `GOOGLE_API_KEY`. It is free, and we use it only for embeddings

**On Google Colab:** click the key icon (Secrets) on the left, add the keys, and switch **Notebook access** on. **On your computer:** put them in a `.env` file.

> **Tips.** The free Groq plan allows about 8,000 tokens a minute. If a cell pauses, it is waiting for the limit. If a cell ends with `RateLimitError`, wait a minute and run it again. If a check ends with `NameError`, you have not run the TODO cell above it yet. Google's free plan allows 100 embedding requests a minute, and every chunk is one request: if you see `RESOURCE_EXHAUSTED`, wait a minute and run the cell again.

In [ ]:
%pip install -qU langchain langgraph langchain-groq langchain-google-genai langchain-text-splitters python-dotenv requests

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()                                                   # on your computer: reads the keys from a .env file
try:
    from google.colab import userdata                           # on Google Colab: reads the keys from Secrets
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass

print("GROQ_API_KEY found:", "GROQ_API_KEY" in os.environ)
print("GOOGLE_API_KEY found:", "GOOGLE_API_KEY" in os.environ)

In [ ]:
from langchain.chat_models import init_chat_model

# One model for the whole week. max_retries=10: when the free plan's speed limit is reached, wait and try again.
llm = init_chat_model("groq:openai/gpt-oss-120b", temperature=0, max_retries=10)

from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.tools import tool

---
# Part 0 - Without RAG

First, ask the model about our course **without** the notes. It has never seen them.

In [ ]:
QUESTION = "In the Advanced Generative AI course, after how many attempts does the evaluator loop stop, and above which score does Prompt Guard block a message?"

no_rag = llm.invoke(QUESTION)
print(no_rag.content)

The right answers are in our notes: **3 attempts**, and a score **above 0.5**. The model can only guess, and a guess written with confidence is the most dangerous kind of answer.

---
# Part 1 - Load and split the notes

The notes are one Markdown file of about 2,000 words (12,000 characters). A search should return a **small piece** that answers the question, not the whole file. So we **split** it into **chunks**.

`RecursiveCharacterTextSplitter` cuts the text into chunks of about `chunk_size` characters. It tries to cut between paragraphs first, then between lines, then between words. `chunk_overlap` repeats a little text between neighbours, so a sentence cut in half is still whole in one of them.

In [ ]:
# Given: download the notes.
import requests

NOTES_URL = "https://raw.githubusercontent.com/AjaySharma78/Advgenai/main/week-3/course_notes.md"
text = requests.get(NOTES_URL).text
print(len(text), "characters")
print(text[:300])

In [ ]:
# TODO 1: make one Document from the text, then split it into chunks.
#   from langchain_core.documents import Document
#   from langchain_text_splitters import RecursiveCharacterTextSplitter
#
#   docs = [Document(page_content=text, metadata={"source": "course_notes.md"})]
#   splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
#   chunks = splitter.split_documents(docs)

In [ ]:
# CHECK 1
print(len(chunks), "chunks")
print("Chunk 5:", chunks[5].page_content)
print("Its metadata:", chunks[5].metadata)

if 20 <= len(chunks) <= 60:
    print("PASS: the notes are split into", len(chunks), "chunks.")
else:
    print("NOT YET: expected about 35 chunks. Check chunk_size=500 and chunk_overlap=100.")

---
# Part 2 - Embeddings and a vector store

An **embedding** turns a text into a long list of numbers. Texts with a **similar meaning** get **similar numbers**, even when they use different words: "How much can I use the free plan?" lands close to "8,000 tokens a minute". This is what makes search by **meaning** possible.

Groq has no embedding model, so we use Google's free one, `gemini-embedding-001`. A **vector store** keeps every chunk with its numbers, and `similarity_search` returns the chunks closest to a question.

In [ ]:
# TODO 2: the embedding model. Then look at one embedding.
#   from langchain_google_genai import GoogleGenerativeAIEmbeddings
#
#   embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
#   vector = embeddings.embed_query("What is a checkpointer?")
#   print(len(vector), "numbers. The first five:", vector[:5])

In [ ]:
# TODO 3: a vector store with all the chunks.
#   from langchain_core.vectorstores import InMemoryVectorStore
#
#   vector_store = InMemoryVectorStore(embeddings)
#   vector_store.add_documents(chunks)                  (this embeds every chunk: a few seconds)

In [ ]:
# CHECK 2: search by meaning. The question uses other words than the notes.
results = vector_store.similarity_search("How much can I use the free Groq plan each minute?", k=2)
for doc in results:
    print("-", doc.page_content[:200].replace("\n", " "))
    print()

if "8,000" in results[0].page_content or "8,000" in results[1].page_content:
    print("PASS: the search found the chunk about 8,000 tokens a minute.")
else:
    print("NOT YET: the 8,000 tokens chunk should be in the top 2. Check that you added the chunks to the vector store.")

---
# Part 3 - The retriever as a tool

Now the search becomes a **tool**, exactly as in the docs: a function that searches the vector store and returns the text of the closest chunks. The **docstring** tells the agent when to use it.

In [ ]:
# TODO 4: the search tool.
#   @tool
#   def search_course_notes(query: str) -> str:
#       """Search the notes of the Advanced Generative AI course (Weeks 1 and 2). Use it for any question about the course."""
#       results = vector_store.similarity_search(query, k=3)
#       texts = []
#       for doc in results:
#           texts.append(doc.page_content)
#       return "\n\n".join(texts)

In [ ]:
# TODO 5: the RAG agent.
#   rag_agent = create_agent(
#       llm,
#       tools=[search_course_notes],
#       system_prompt="You are the assistant of the Advanced Generative AI course. For any question about the course, call search_course_notes "
#                     "and answer ONLY from what it returns. If the notes do not say, answer that the notes do not say. For greetings, answer directly.",
#   )

In [ ]:
# CHECK 3: the course question needs the tool; a greeting does not.
def used_search(result):
    for message in result["messages"]:
        if message.type == "tool":
            return True
    return False

rag_result = rag_agent.invoke({"messages": QUESTION})
print("Course question:", rag_result["messages"][-1].content)
print()
hello = rag_agent.invoke({"messages": "Hi! How are you today?"})
print("Greeting:", hello["messages"][-1].content)
print()

answer = rag_result["messages"][-1].content.lower()
if used_search(rag_result) and ("3" in answer or "three" in answer) and "0.5" in answer:
    print("PASS: the agent searched the notes and gave both right answers.")
else:
    print("NOT YET: the answer should say 3 attempts and 0.5, from the notes. Check the tool and the system prompt.")
if not used_search(hello):
    print("PASS: the agent did not search for a greeting.")
else:
    print("NOT YET (it can happen): the agent searched for a greeting. Make the system prompt clearer.")

---
# Part 4 - Is the answer grounded?

RAG lowers guessing, but it does not remove it. As in Week 2 Day 3, an **evaluator** checks the answer against the **chunks**: is every fact in them? We grade the RAG answer and the answer from Part 0.

In [ ]:
# TODO 6: the grounding evaluator.
#   class Grounded(BaseModel):
#       grounded: bool = Field(description="True only if every fact in the answer is stated in the notes")
#       reason: str = Field(description="One sentence that explains the grade")
#
#   grader = llm.with_structured_output(Grounded, method="json_schema", strict=True).with_retry(stop_after_attempt=3)

In [ ]:
# CHECK 4: grade both answers against the same chunks.
notes_for_question = search_course_notes.invoke({"query": QUESTION})

with_rag = grader.invoke(f"Notes:\n{notes_for_question}\n\nAnswer:\n{rag_result['messages'][-1].content}")
without_rag = grader.invoke(f"Notes:\n{notes_for_question}\n\nAnswer:\n{no_rag.content}")
print("With RAG   :", with_rag.grounded, "-", with_rag.reason)
print("Without RAG:", without_rag.grounded, "-", without_rag.reason)

if with_rag.grounded:
    print("PASS: the RAG answer is grounded in the notes.")
else:
    print("NOT YET: read the reason. Is the answer adding facts the notes do not have?")

In [ ]:
# TODO 7: answer in your own words.
#   why_not_whole_file = "..."     (two sentences: the notes are about 3,000 tokens. Why not put the whole file in the system prompt instead of using RAG? When would that be fine?)

In [ ]:
# CHECK 5
if len(why_not_whole_file.split()) >= 20:
    print("PASS")
else:
    print("NOT YET: write two sentences.")

---
# Today you learned
- **RAG** puts only the pieces of your documents that matter into the context.
- **Split** with `RecursiveCharacterTextSplitter`; **embed** with `GoogleGenerativeAIEmbeddings`; **store and search** with `InMemoryVectorStore`.
- **Agentic RAG**: the retriever is a **tool**, and the agent decides when to use it.
- An **evaluator** checks that the answer is grounded in the chunks.

**Next class: MCP and the project.** The study assistant joins the notes, the memory of Day 2, and the official LangChain docs through MCP.